# S1 — IS diagnosis (sealed OOS off-limits)

Forensics on the **frozen** STAR using **research IS only** (`is_research_is`).

Look for tails (excess kurtosis), cost drag, concentration, and fold-level instability **before** opening `09_oos_tearsheet.ipynb`. Disposition: kill / paper / new H. Do not re-tune on sealed OOS.


## 0. Imports & Config


In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from dataclasses import replace

from backtest.s1_equities.research import (
    DEFAULT_STAR_STACK,
    load_star_stack,
    n_trials_ledger_total,
    vt_target_ann_vol_from_stack,
)
from backtest.s1_equities.signals import (
    default_backtest_paths,
    is_mask_from_preds,
    load_ohlc_panels,
    load_predictions,
    prediction_path_for_timing,
    score_matrix,
)
from backtest.s1_equities.runner import run_backtest, summarize_periods
from backtest.s1_equities.stops import parse_stop_star
from performance.sharpe_inference import return_moments
from risk.analytics.s1_equities.vol_targeting import parse_vol_target_star

stack = load_star_stack(DEFAULT_STAR_STACK)
print("STAR", {k: stack[k] for k in stack if k.endswith("_STAR")})
print("variant ledger arms", n_trials_ledger_total())
print("Disposition: kill / paper / new H — never re-tune on sealed OOS")


## 1. Run frozen STAR on research IS only


In [ ]:
paths = default_backtest_paths(ROOT)
timing = str(stack["TIMING_STAR"])
pred_path = prediction_path_for_timing(paths, timing)
if not os.path.isfile(pred_path):
    raise FileNotFoundError(
        f"IS predictions missing: {pred_path}. Build them before this diagnosis."
    )
preds = load_predictions(pred_path)
is_mask = is_mask_from_preds(preds)
assert bool(is_mask.any()), "research IS mask is empty"
scores = score_matrix(preds)
opens, highs, lows, closes = load_ohlc_panels(paths["features"], tickers=list(scores.columns))
vt = replace(
    parse_vol_target_star(str(stack["VT_STAR"])),
    target_ann_vol=float(vt_target_ann_vol_from_stack(stack)),
)
res = run_backtest(
    scores,
    opens,
    closes,
    n=int(stack["N_STAR"]),
    timing_mode=timing,
    weight_mode=str(stack["WEIGHT_STAR"]),
    inv_vol_window=int(stack.get("INV_VOL_WINDOW_STAR", 42)),
    highs=highs,
    lows=lows,
    stop=parse_stop_star(str(stack["STOP_STAR"])),
    date_mask=is_mask,
    vol_target=vt,
    max_gross=float(stack.get("MAX_GROSS_STAR", 1.5)),
)
is_ret = res.period_returns
print("IS periods", int(is_ret.shape[0]), "first", is_ret.index.min(), "last", is_ret.index.max())


## 2. Headline IS metrics and tails


In [ ]:
stats = summarize_periods(is_ret)
mom = return_moments(is_ret, periods_per_year=52.0)
excess_kurt = float(mom["kurtosis"] - 3.0) if np.isfinite(mom["kurtosis"]) else float("nan")
row = {
    "ann_sharpe": stats["sharpe"],
    "cagr": stats["cagr"],
    "max_drawdown": stats["max_drawdown"],
    "vol": stats["vol"],
    "skew": mom["skew"],
    "excess_kurtosis": excess_kurt,
    "psr_input_sr_period": mom["sr_period"],
}
display(pd.Series(row, name="IS"))
if np.isfinite(excess_kurt) and excess_kurt > 5:
    warnings.warn(f"IS excess kurtosis {excess_kurt:.2f} — inspect tails before opening OOS")
if np.isfinite(stats["max_drawdown"]) and stats["max_drawdown"] < -0.25:
    warnings.warn(f"IS max DD {stats['max_drawdown']:.1%} breaches a 25% hole")


## 3. Cause-class checklist

- Extreme single-name concentration?
- Excess kurtosis from a handful of weeks?
- Costs eating a thin gross edge?
- Vol target fit only on IS (see `06_risk/notebooks/s1_equities/01_leverage.ipynb`)?

If any item is a kill, stop. Do not open `09_oos_tearsheet.ipynb` to "check".
